In [ ]:
import os
import json
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import Literal

print("✅ Reviewer agent libraries imported.")

In [ ]:
# Strict structure the reviewer MUST return
class ReviewResult(BaseModel):
    verdict: Literal["APPROVE", "REVISE", "REJECT"] = Field(
        description="Final decision on the RAG answer quality"
    )
    score: int = Field(
        description="Quality score from 0 to 10",
        ge=0, le=10
    )
    factual_grounding: str = Field(
        description="Are all claims supported by the retrieved context?"
    )
    completeness: str = Field(
        description="Were all parts of the user question answered?"
    )
    clinical_safety: str = Field(
        description="Any diagnosis, dosage, or clinical boundary violations?"
    )
    hallucination_flags: list[str] = Field(
        description="List any terms, muscles, or claims NOT found in context"
    )
    issues: list[str] = Field(
        description="List of all identified problems (empty if APPROVE)"
    )
    revised_answer: str = Field(
        description="Corrected answer if verdict is REVISE, else empty string"
    )
    reviewer_notes: str = Field(
        description="Short explanation of the final verdict"
    )

parser = JsonOutputParser(pydantic_object=ReviewResult)
print("✅ Review schema defined.")

In [ ]:
print("🧠 Initializing Reviewer — Llama 3.3 70B...")

# Intentionally DIFFERENT from the RAG agent
# Two different models = two different blind spots = better coverage
reviewer_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=2048,
)

reviewer_system_prompt = """
You are **MedReview AI** — a strict, expert-level quality control agent for a \
Sports Medicine and Fitness AI assistant.

Your ONLY job is to critically evaluate answers produced by a RAG (Retrieval-Augmented \
Generation) system and decide: APPROVE, REVISE, or REJECT.

You will receive three inputs:
  1. The original user question
  2. The RAG agent's answer
  3. The retrieved context chunks the RAG agent had access to

══════════════════════════════════════════
         YOUR REVIEW CHECKLIST
══════════════════════════════════════════

Check each dimension carefully:

🔬 FACTUAL GROUNDING
   → Every claim in the answer must be traceable to the retrieved context.
   → Flag anything stated with confidence that is NOT in the context.

📋 COMPLETENESS
   → If the user asked multiple questions, all must be addressed.
   → Flag any sub-question that was skipped or only partially answered.

⚕️ CLINICAL SAFETY
   → Flag any medication names or dosages recommended.
   → Flag any direct injury diagnoses made.
   → Flag any advice that replaces individualized professional assessment.

🚨 HALLUCINATION SCAN
   → List every muscle name, anatomical term, or clinical claim
     that appears in the answer but NOT in the retrieved context.

══════════════════════════════════════════
         VERDICT CRITERIA
══════════════════════════════════════════

✅ APPROVE  → Score 8–10
   • All claims grounded in context
   • All sub-questions answered
   • No clinical safety violations
   • No hallucinations detected

⚠️ REVISE   → Score 4–7
   • Minor gaps, missing details, or slightly ungrounded claims
   • Provide a corrected, improved version in revised_answer
   • Be specific about what you changed and why

❌ REJECT   → Score 0–3
   • Major hallucinations detected
   • Clinical safety violations present
   • Context was empty but model answered anyway
   • Answer is dangerously incomplete or misleading

══════════════════════════════════════════
         OUTPUT FORMAT — CRITICAL
══════════════════════════════════════════

You MUST respond with ONLY a valid JSON object. No preamble, no explanation outside JSON.
No markdown code fences. Raw JSON only.

{format_instructions}
"""

reviewer_prompt = ChatPromptTemplate.from_messages([
    ("system", reviewer_system_prompt),
    ("human",
     "══════════════════════════════\n"
     "📌 ORIGINAL USER QUESTION:\n{question}\n\n"
     "══════════════════════════════\n"
     "🤖 RAG AGENT ANSWER:\n{rag_answer}\n\n"
     "══════════════════════════════\n"
     "📚 RETRIEVED CONTEXT CHUNKS:\n{context}\n"
     "══════════════════════════════\n"
     "Now perform your full review and return ONLY the JSON verdict."
    ),
]).partial(format_instructions=parser.get_format_instructions())

reviewer_chain = reviewer_prompt | reviewer_llm | parser
print("✅ Reviewer chain assembled.")

In [ ]:
def review_answer(question: str, rag_answer: str, context_docs: list) -> dict:
    """Runs the RAG output through the reviewer agent."""
    context_text = "\n\n---\n\n".join([
        f"[Chunk {i+1}]\n{doc.page_content}"
        for i, doc in enumerate(context_docs)
    ]) if context_docs else "⚠️ No context was retrieved."

    print("🔍 Reviewer agent evaluating answer...")
    return reviewer_chain.invoke({
        "question":   question,
        "rag_answer": rag_answer,
        "context":    context_text,
    })

def print_review(review: dict):
    verdict = review.get("verdict", "UNKNOWN")
    score   = review.get("score", 0)
    badge = {"APPROVE": "✅", "REVISE": "⚠️", "REJECT": "❌"}.get(verdict, "❓")

    print(f"\n{'━'*60}")
    print(f"  {badge}  VERDICT: {verdict}   |   SCORE: {score}/10")
    print(f"{'━'*60}")
    print(f"\n🔬 Factual Grounding:\n   {review.get('factual_grounding')}")
    print(f"\n📋 Completeness:\n   {review.get('completeness')}")
    print(f"\n⚕️  Clinical Safety:\n   {review.get('clinical_safety')}")

    flags = review.get("hallucination_flags", [])
    print(f"\n🚨 Hallucination Flags: {'None' if not flags else ''}")
    for f in flags: print(f"   • {f}")

    issues = review.get("issues", [])
    print(f"\n⚠️  Issues Found: {'None' if not issues else ''}")
    for issue in issues: print(f"   • {issue}")

    print(f"\n📝 Reviewer Notes:\n   {review.get('reviewer_notes')}")
    if verdict == "REVISE":
        print(f"\n✏️  REVISED ANSWER:\n{'─'*60}")
        print(review.get("revised_answer", ""))
    print(f"\n{'━'*60}\n")

def ask_with_review(question: str):
    """
    Full pipeline:
    1. Medical RAG agent answers
    2. Reviewer agent evaluates
    3. Returns best available answer
    """
    print(f"\n{'━'*60}")
    print(f"❓ Question: {question}")
    print(f"{'━'*60}")

    print("\n📡 Step 1: Medical RAG Agent retrieving & answering...")
    # NOTE: Calling the medical_rag_chain established in the previous node!
    rag_result  = medical_rag_chain.invoke({"input": question})
    rag_answer  = rag_result["answer"]
    context_docs = rag_result.get("context", [])

    print("\n🤖 RAG Answer:\n")
    print(rag_answer)

    print("\n🔍 Step 2: Reviewer Agent evaluating...")
    review = review_answer(question, rag_answer, context_docs)
    print_review(review)

    verdict = review.get("verdict")
    print(f"\n{'━'*60}\n📤 FINAL ANSWER DELIVERED:\n{'━'*60}\n")

    if verdict == "APPROVE":
        print("✅ Original answer approved — no changes needed.\n")
        print(rag_answer)
    elif verdict == "REVISE":
        print("⚠️ Reviewer improved the answer:\n")
        print(review.get("revised_answer"))
    elif verdict == "REJECT":
        print("❌ Answer rejected by reviewer.")
        print("Please rephrase your question or check the knowledge base.\n")

    return {
        "question":       question,
        "rag_answer":     rag_answer,
        "review":         review,
        "final_answer":   review.get("revised_answer") if verdict == "REVISE" else rag_answer,
        "verdict":        verdict,
        "score":          review.get("score"),
    }